In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.6 MB/s eta 0:00:00


In [ ]:
import math
import pandas as pd
import re
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
vowels = set("аеёиоуыэюяАЕЁИОУЫЭЮЯ")

def count_vowels(w):
    return sum(ch in vowels for ch in w)

def extract_features_from_text(text):
    syntagms = re.split(r'([,.!?;:—])', text)

    final_syntagms = []
    for i in range(0, len(syntagms) - 1, 2):
        synt = syntagms[i].strip()
        punct = syntagms[i+1]
        if synt:
            final_syntagms.append((synt, punct))

    if len(syntagms) % 2 == 1 and syntagms[-1].strip():
        final_syntagms.append((syntagms[-1].strip(), None))

    rows = []
    synt_num = 0

    for synt, end_punct in final_syntagms:
        synt_num += 1

        words = re.findall(r'\b\w+\b', synt, flags=re.UNICODE)
        synt_len = len(words)

        for i, w in enumerate(words):
            prev_word = words[i - 1] if i > 0 else None
            next_word = words[i + 1] if i < len(words) - 1 else None
            punct_after = end_punct if i == len(words) - 1 else None
            if punct_after and punct_after in "-':;":
                punct_after = ','
            word = w + punct_after if punct_after else w

            rows.append({
                "word": word,
                "word_length": len(w),
                "previous_word": prev_word,
                "next_word": next_word,
                "punct_after": punct_after,
                "syntagma_length": synt_len,
                "synt_num": synt_num,
                "part_of_speech": 0,     # нет данных → 0
                "word_form": 0,
                "word_genesys": 0,
                "word_semantics1": 0,
                "word_semantics2": 0,
                "words_before": i,
                "words_after": synt_len - i - 1,
                "sent_words_before": i,
                "sent_words_after": synt_len - i - 1,
                "vowels_in_word": count_vowels(w),
                "vowels_before": sum(count_vowels(x) for x in words[:i]),
                "vowels_after": sum(count_vowels(x) for x in words[i+1:]),
                "capital_letter": int(w[0].isupper()),
                "intonation": 0,
                "punctuation": punct_after,
                "pause_len": 0
            })

    # 4. Делаем DataFrame
    df = pd.DataFrame(rows)
    return df


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, num_heads, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, num_heads*d_k)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))

        cross_attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm3(x + self.dropout(cross_attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout, device):
        super(Transformer, self).__init__()
        self.device = device
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2).to(self.device)
        tgt_pad_mask = (tgt != 0).unsqueeze(1).unsqueeze(2).to(self.device)
        seq_length = tgt.size(1)
        nopeak_mask = torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1).bool().to(self.device)

        tgt_mask = tgt_pad_mask & (~nopeak_mask.unsqueeze(0))

        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

In [ ]:
import joblib

# Load the pause regression model
clf_pause = joblib.load('clf_pause_model.pkl')

allophone_model = torch.load('model.pth', weights_only=False)


In [ ]:
def greedy_decode(model, input_text, input_tokenizer, target_tokenizer, max_len=50, device="cpu"):
    model.eval()

    # Токенизация входа
    src_ids = torch.tensor(input_tokenizer.encode(input_text, max_length=max_len)).unsqueeze(0).to(device)  # (1, seq_len)
    # <BOS> для выхода
    generated = torch.tensor([target_tokenizer.phone2id["<BOS>"]], device=device).unsqueeze(0)  # (1,1)
    with torch.no_grad():
        for _ in range(max_len):
            # Предсказание
            output = model(src_ids, generated)  # (1, seq_len, vocab_size)
            next_token_logits = output[0, -1, :]  # логиты последнего токена
            next_token_id = next_token_logits.argmax().item()

            if next_token_id == target_tokenizer.phone2id["<EOS>"]:
                break

            # Добавляем токен к последовательности
            generated = torch.cat([generated, torch.tensor([[next_token_id]], device=device)], dim=1)

    # Декодируем в список аллофонов
    pred_phones = target_tokenizer.decode(generated[0].tolist())
    return pred_phones

In [ ]:
class CharTokenizer:
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.char2id = {c: i+4 for i, c in enumerate(chars)}  # +2 для PAD/BOS
        self.char2id["<PAD>"] = 0
        self.char2id["<BOS>"] = 1
        self.char2id["="] = 2
        self.char2id["|"] = 3
        self.char2id["<EOS>"] = len(self.char2id)
        self.id2char = {v: k for k, v in self.char2id.items()}

    def encode(self, text, add_special_tokens=True, max_length=50):
        ids = [self.char2id.get(c, 0) for c in text]
        if add_special_tokens:
            ids = [self.char2id["<BOS>"]] + ids + [self.char2id["<EOS>"]]
        if len(ids) < max_length:
            ids += [self.char2id["<PAD>"]] * (max_length - len(ids))
        return ids[:max_length]

    def decode(self, ids):
        return "".join(self.id2char[i] for i in ids if i not in (0,1,self.char2id["<EOS>"]))

class PhoneTokenizer:
    def __init__(self, sequences):

        # уникальные аллофоны
        unique_phones = sorted(set(phone for seq in sequences for phone in seq))
        self.phone2id = {p: i+2 for i, p in enumerate(unique_phones)}  # +2 для PAD=0, BOS=1
        self.phone2id["<PAD>"] = 0
        self.phone2id["<BOS>"] = 1
        self.phone2id["<EOS>"] = len(self.phone2id)

        self.id2phone = {v: k for k, v in self.phone2id.items()}

    def encode(self, seq, add_special_tokens=True, max_length=50):
        ids = [self.phone2id.get(p, 0) for p in seq]  # 0 = PAD/UNK
        if add_special_tokens:
            ids = [self.phone2id["<BOS>"]] + ids + [self.phone2id["<EOS>"]]
        # паддинг до max_length
        if len(ids) < max_length:
            ids += [self.phone2id["<PAD>"]] * (max_length - len(ids))
        return ids[:max_length]

    def decode(self, ids):
        phones = []
        for i in ids:
            if i in (0, 1):  # PAD или BOS
                continue
            if self.id2phone[i] == "<EOS>":
                break
            phones.append(self.id2phone[i])
        return phones

# Преобразование текстов в траскрипцию

In [ ]:
# Тексты для теста
texts = ["Он искал ее в Геленджике, в Гаграх, в Сочи.",
         "На другой день по приезде в Сочи,он купался утром в море, потом брился, надел чистое белье, белоснежный китель,\
       позавтракал в своей гостинице на террасе ресторана, выпил бутылку шампанского, пил кофе с шартрезом,\
        не спеша выкурил сигару.",
         "Возвратясь в свой номер, он лег на диван и выстрелил себе в виски из двух револьверов.",
         "Я ни перед чем не остановлюсь, защищая свою честь, честь мужа и офицера!",
         "А зачем все делается на свете?",
         " Разве мы понимаем что-нибудь в наших поступках?",

          "В анамнезе супермена был синтезированный компьютером криптонит.",
        "Мы всем отрядом съели все, что было во всем лагере, и все остались довольны.",
         "Бык тупогуб, тупогубенький бычок, У быка бела губа была тупа.",
         "В день Тлакашипеуалицтли жители Теночтитлана славят Шипе-Тотека, но не забывают также и про Уицилопочтли, Кецалькоатля и Тескатлипоку.",
         "Вот оно что. А он что? Да ты что!"]

In [ ]:
input_texts = [''.join(['Т', 'Е', 'М', 'Н', 'Ы', 'А', 'Л', 'И', 'В', 'х', 'о', 'л', 'д', 'н', 'е', 'с', 'а', 'т', 'ь', ',', 'й', 'и', 'з', 'б', 'ш', 'у', 'к', 'р', 'г', 'ж', 'я', 'м', 'ч', 'ы', 'в', 'п', 'ц', 'ю', '.', 'щ', ';', 'I', 'К', '-', ' ', ':', 'П', 'Э', '!', 'ф', 'Д', '?', 'С', 'Х', 'Ч', 'Ж', 'О', 'Я', 'Б', 'э', 'У', 'З', 'ъ', '"', 'Р', 'Г', 'Ш', 'Й', 'Ц', 'R', 'A', 'm', 'a', 't', 'n', 'o', 'b', 'i', 's', 'q', 'u', 'r', 'l', '*', 'Ю', 'B', 'P', 'e', 'p', 'E', 'M', 'c', 'C', 'h', '~', 'd', 'x', 'Ф', '1', '7', 'F', 'z', 'g'])]
allophones_seqs = [["t'", 'o0', 'm', 'n', 'y4', 'j', 'i4', 'a1', "l'", 'e0', 'f', 'h', 'l', 'd', 'a4', "s'", "n'", 'i1', 'a0', 'a2', 'z', 'b', 'sh', 'y0', 't', 'u0', 's', "k'", 'H', 'r', 'k', 'zh', "d'", "m'", "r'", "g'", 'ch', 'g', 'i0', "b'", 'v', "v'", "z'", 'y1', 'p', 'c', 'u1', "p'", 'u4', 'sc', 'o1', "h'", "f'", 'CH', 'C', 'o4', 'SC', 'e4', 'e1']]

input_tokenizer = CharTokenizer(input_texts)
target_tokenizer = PhoneTokenizer(allophones_seqs)

In [ ]:
texts_with_pause = []
for t in texts:
    features = extract_features_from_text(t) # извлечение признаков по тексту

    # применяем pipeline
    pred_len = clf_pause.predict(features) # модель с паузами
    final_text = []
    for i in range(len(pred_len)):

      prev_word = features['previous_word'][i] if features['previous_word'][i] else '-'
      next_word = features['next_word'][i] if features['next_word'][i] else '-'
      current_word = features['word'][i]

      input_text = (
        f"{current_word} | pre={prev_word} | "
        f"next={next_word} | part={'1'}"
    )
      allophones = greedy_decode(allophone_model, # модель с аллофонами
                                 input_text,
                                 input_tokenizer,
                                 target_tokenizer,
                                 device=device)
      if features['punct_after'][i]:
        allophones.append(f' {features['punct_after'][i]}')


      final_text.append(''.join(allophones))
      if pred_len[i] >= 100:
        final_text.append('|'*int(pred_len[i]/200)) # спец симфол для паузы
    texts_with_pause.append(' '.join(final_text))
texts_with_pause


["o0n i1ska0l ji1jo0 v g'i1l'i1ndzhy0sk'i4 , |||| va0 ga0gra4h , ||| f so0chi4 . ||||",
 "na2 dru1go0j d'e0n' pa2 pr'i1je0z'd'i4 f so0chi4 , |||| o0n ku1pa0ls'a4 u0tra4m vm mo0r'i4 , |||| pa1to0m br'i0ls'a4 , ||| na1d'e0l chi0sta4ji4 b'i1l'jo0 , |||| b'i1la1s'n'e0zhny4j k'i0t'i4l' , ||| pa1za0ftra4ka4l f sva1je0j ga2s't'i1n'i0cy4 na2 t'i1ra0s'i4 r'i1sta1ra0na4 , ||| vy0p'i4l bu1ty0lku4 sha1mpa0nska4va4 , |||| p'i0l ko0f'i4 s sha1rtr'e0za4m , |||| n'i1 sp'i1sha0 vy0ku4r'i4l s'i1ga0ru4 . ||||",
 "va2zvra1t'a0s' v svo0j no0m'i4r , |||| o0n l'o0k na2 d'i0va4n y1 vy0str'i4l'i4l s'i1b'e0 f v'i0sk'i4 y1z dvu0h r'i1vo0l'va4 . ||||||",
 "ja0 n'i1 p'i1r'i1d che0m n'i1 a1sta1no0vl'u4s' , |||| za2sci1sca0ja4 sva1ju0 che0s't' , |||| che0s't' mu0zha4 i1 a1f'i1ce0ra4 ! ||||",
 "a1 za1che0m fs'o0 d'e0la4ji4ca4 na2 sv'e0t'i4 ? ||||",
 "ra0zv'i4 my0 pa2n'i1ma0ji4m shto0 n'i1bu0t' v na0shy4h pa2stu1pka0h ? |||",
 "v a1na0mn'i4z' su1p'i1rm'e0na4 by0l s'i1n't'i1z'i1ra1va0jny4j ka1m'p'ju0t'i4ra4m kr'i1pto0n

In [ ]:
import csv
from tqdm import tqdm

## Преобразование текстового файла

In [ ]:
result_metadata = []

In [ ]:
with open('metadata_RUSLAN_22200.csv') as f:
  csvreader = csv.reader(f, delimiter='|')
  for row in tqdm(csvreader):
      wav, text = row[0], row[1]

      features = extract_features_from_text(text)

      # применяем pipeline
      pred_len = clf_pause.predict(features)
      final_text = []

      for i in range(len(pred_len)):


        prev_word = features['previous_word'][i] if features['previous_word'][i] else '-'
        next_word = features['next_word'][i] if features['next_word'][i] else '-'
        current_word = features['word'][i]

        input_text = (
          f"{current_word} | pre={prev_word} | "
          f"next={next_word} | part={'1'}"
      )
        allophones = greedy_decode(allophone_model,
                                  input_text,
                                  input_tokenizer,
                                  target_tokenizer,
                                  device=device)

        if features['punct_after'][i]:
          allophones.append(f' {features['punct_after'][i]}')

        final_text.append(''.join(allophones))

        if pred_len[i] >= 100:
          final_text.append('-'*int(pred_len[i]/200))

      texts_with_pause = ' '.join(final_text)

      result_metadata.append({'wav': wav,
                              'text': text,
                              'transcription': texts_with_pause})



FileNotFoundError: [Errno 2] No such file or directory: 'metadata_RUSLAN_22200.csv'

In [ ]:
result_metadata

In [ ]:
fieldnames = ['wav', 'text', 'transcription']
with open('metadata_ruslan.csv', 'w', newline='', encoding='utf-8', delimiter='|') as csvfile:
    # Создаем объект DictWriter
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # Записываем заголовки
    writer.writeheader()

    # Записываем строки из словарей
    writer.writerows(result_metadata)


Сохранение нового файла

In [ ]:
with open('metadata_ruslan.txt') as f:
  for str in result metadata:
    f.write(f'{str['wav']}|{str['text']}|{str['transcription']}/n')